# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainUlAbideen02/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = One unique content page (content_id) for a specific client (client_id).

Time Window: Historical performance metrics calculated over a 90-day lookback window (impressions_90d, clicks_90d) evaluated on a mid-panel observation month.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess, pandas as pd, numpy as np

# Clone repo data if not locally present in Colab
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Working Directory: {os.getcwd()}")
print(f"Dataset Verified: {os.path.exists(data_path)}")
print(f"Total Rows: {len(df):,}, Unique content_id count: {df['content_id'].nunique():,}")

Working Directory: /content/flyrank-ml-internship-starter
Dataset Verified: True
Total Rows: 30,000, Unique content_id count: 30,000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: search_volume, competition, cpc, word_count, char_count, avg_position, ctr, engagement_rate, scroll_rate (All knowable prior to prediction time).

Label: is_declining (Binary target: 1 if trend_direction == 'down', else 0).

Context: content_id, client_id, content_type, main_intent, impression_tier, position_tier.

Excluded (and why): trend_pct and direct future click delta columns are excluded from features because they represent future outcome measurements (data leakage) that are unknowable at prediction time.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Construct the feature frame and label
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

feature_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", "avg_position", "ctr", "engagement_rate", "scroll_rate"]
X_features = df[feature_cols].copy()
y_label = df["is_declining"]

print("Feature Bucket Shape:", X_features.shape)
print("Label Distribution:\n", y_label.value_counts(normalize=True).round(3))

Feature Bucket Shape: (30000, 9)
Label Distribution:
 is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Queries:

Grain Check: Validates that content_id uniquely identifies each row.

Counts & Availability: Checks row count, active rows where impressions_90d > 0 IS TRUE, and missing values across key features.

Leakage Verification: Compares model metrics with and without the excluded trend_pct column to verify data hygiene.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Grain Check
is_unique_grain = df["content_id"].nunique() == len(df)
print(f"Query 1 — Grain is unique per row: {is_unique_grain}")

# Query 2: Active Row Availability & Null Checks
active_rows = len(df[df["impressions_90d"] > 0])
missing_count = df[feature_cols].isna().sum().sum()
print(f"Query 2 — Active Rows (impressions_90d > 0 IS TRUE): {active_rows:,} ({active_rows/len(df)*100:.1f}%)")
print(f"Query 2 — Missing values in feature set: {missing_count}")

# Query 3: Leakage Test (The Trap)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Honest Model
X_honest = X_features.fillna(0)
rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_honest, y_label)
honest_pred = rf.predict(X_honest)
print(f"Query 3 — Honest Precision@50: {precision_score(y_label, honest_pred):.3f}")

# Leaked Model (The Trap)
X_leaked = X_honest.copy()
X_leaked["LEAK_trend_pct"] = df["trend_pct"]
rf.fit(X_leaked, y_label)
leaked_pred = rf.predict(X_leaked)
print(f"Query 3 — Leaked Precision@50 (The Trap): {precision_score(y_label, leaked_pred):.3f} (Artificial perfect score)")
del X_leaked

Query 1 — Grain is unique per row: True
Query 2 — Active Rows (impressions_90d > 0 IS TRUE): 30,000 (100.0%)
Query 2 — Missing values in feature set: 22927
Query 3 — Honest Precision@50: 0.987
Query 3 — Leaked Precision@50 (The Trap): 1.000 (Artificial perfect score)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitations:

Window Overlaps: Performance metrics rely on aggregated 90-day windows (impressions_90d), which smooth out temporary traffic spikes or short-lived Google algorithm tests.

Cold Start Pages: Pages younger than 90 days or pages with zero impressions lack historical trend metrics and cannot be accurately scored by this refresh pipeline.

Search Console Scope: Search Console data only reflects user demand on Google Search and does not account for direct, referral, or social media traffic shifts.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify cold start / low age limit in dataset
young_pages = len(df[df["content_age_days"] < 90])
zero_imp_pages = len(df[df["impressions_90d"] == 0])

print(f"Limitation check — Pages younger than 90 days: {young_pages:,}")
print(f"Limitation check — Zero impression pages excluded from refresh logic: {zero_imp_pages:,}")

Limitation check — Pages younger than 90 days: 0
Limitation check — Zero impression pages excluded from refresh logic: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.